# Hello world — astro-inject

This notebook is the visual sanity check for the scaffolding. It does four things:

1. Imports `astro_inject` and prints the version.
2. Synthesizes a small noisy 512×512 image with a few Gaussian sources.
3. Wraps it as a `CCDData` with a minimal TAN WCS.
4. Renders it inline with `astro_inject.viz.plot_image()` (asinh stretch + percentile interval).

If you can see a recognisably astronomical-looking image at the bottom of the notebook, the plumbing works.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from astropy.nddata import CCDData
from astropy.wcs import WCS

import astro_inject
from astro_inject.viz import plot_image

print(f"astro_inject {astro_inject.__version__}")

## Synthesize a small frame

Gaussian background + a handful of Gaussian point sources at random positions. Reproducible via the explicit `np.random.default_rng` seed.

In [ ]:
rng = np.random.default_rng(seed=2026)
shape = (512, 512)

background = rng.normal(loc=100.0, scale=5.0, size=shape).astype(np.float32)

n_sources = 40
ys = rng.uniform(0, shape[0], size=n_sources)
xs = rng.uniform(0, shape[1], size=n_sources)
amps = rng.lognormal(mean=5.5, sigma=0.7, size=n_sources)
sigmas = rng.uniform(1.5, 2.5, size=n_sources)

yy, xx = np.mgrid[: shape[0], : shape[1]]
image_data = background.copy()
for y0, x0, amp, sigma in zip(ys, xs, amps, sigmas, strict=True):
    image_data += (amp * np.exp(-((yy - y0) ** 2 + (xx - x0) ** 2) / (2 * sigma**2))).astype(
        np.float32
    )

image_data.shape, image_data.dtype, float(image_data.min()), float(image_data.max())

## Wrap as `CCDData` with a minimal WCS

In [ ]:
wcs = WCS(naxis=2)
wcs.wcs.crpix = [shape[1] / 2, shape[0] / 2]
wcs.wcs.crval = [10.0, -20.0]
wcs.wcs.cdelt = [-0.0001, 0.0001]
wcs.wcs.ctype = ["RA---TAN", "DEC--TAN"]

image = CCDData(image_data, unit="adu", wcs=wcs)
image

## Render with `plot_image`

This is the only "real" piece of functionality in the current scaffold — an asinh stretch plus a percentile interval, which is the standard astronomy-friendly default.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
plot_image(image, ax=ax, percentile=99.5)
ax.set_title("astro-inject hello world — synthetic frame")
plt.show()

## What just happened?

* We exercised the full import path of the package.
* We confirmed `CCDData`, WCS, and matplotlib all play together as expected.
* We verified that `plot_image` produces a recognisably astronomical-looking render.

## What's next?

Real injection physics is intentionally **not** implemented yet. The next milestones are:

1. Fill in `astro_inject.trails.TrailParameters` with real geometry and a brightness profile.
2. Implement `astro_inject.composition.inject(...)` so it returns a populated `InjectionResult`.
3. Add the first concrete `Instrument` descriptors and validate against real frames.

Until then, this notebook is the smoke test that the scaffolding is solid.